## Zadanie domowe: BBHE i DSIHE

W klasycznym wyrównywaniu histogramu HE  po wykonaniu operacji jasność obrazu ulega zmianie.
Dało się to zaobserwować podczas przeprowadzonych eksperymentów.
Jeśli nie to należy uruchomić skrypt z sekcji A i zwrócić na to uwagę.
Średnia jasność dąży do środkowego poziomu szarości.
Jest to wada i dlatego klasyczne HE ma ograniczone zastosowanie.

Powstało sporo metod, które eliminują to niekorzystne zjawisko.
Najprostsze z nich polegają na dekompozycji obrazu wejściowego na dwa podobrazy (wg. pewnego kryterium).
Następnie operacja HE wykonywana jest dla tych podobrazów.

Dwie znane z literatury metody to:
- Bi-Histogram Equalization
- DSIHE - Dualistic Sub-Image Histogram Equalization

W metodzie BBHE za kryterium podziału przyjmuje się średnią jasność w obrazie.
W DSIHE obraz dzieli się na dwa podobrazy o takiej samej liczbie pikseli (jaśniejszych i ciemniejszych).

W ramach zadania należy zaimplementować wybraną metodę: BBHE lub DSIHE (ew. obie).

1. Wczytaj obraz *jet.bmp* i wylicz jego histogram.
2. W kolejnym kroku należy wyznaczyć próg podziału obrazu na dwa podobrazy (*lm*).
3. Dla BBHE wyznacz średnią jasność obrazu. Dla DSIHE można wykorzystać histogram skumulowany.
Należy znaleźć poziom jasności który znajduje się "w połowie" histogramu skumulowanego.
W tym celu warto stworzyć tablicę, zawierającą moduł histogramu skumulowanego pomniejszonego o połowę liczby pikseli.
Następnie znaleźć minimum.
4. Dalej należy podzielić histogram oryginalnego obrazu na dwa histogramy *H1* i *H2*.
Dla każdego z nich wyliczyć histogram skumulowany ($C_1$ i $C_2$) i wykonać normalizację.
Normalizacja polega na podzieleniu każdego histogramu przez jego największy element.
5. Na podstawie histogramów skumulowanych należy stworzyć przekształcenie LUT.
Należy tak przeskalować $C_1$ i $C_2$, aby uzyskać jednorodne przekształcenie.
Tablicę $C_1$ wystarczy pomnożyć przez próg podziału.
Tablicę $C_2$ należy przeskalować do przedziału: $<lm+1; 255>$, gdzie $lm$ jest progiem podziału.<br>
$C_{1n} = (lm)*C1;$<br>
$C_{2n} = lm+1 + (255-lm-1)*C2;$<br>
Następnie dwie części tablicy przekodowań należy połączyć.
6. Ostatecznie należy wykonać operację LUT i wyświetlić wynik wyrównywania histogramu.
Porównaj wynik operacji BBHE lub DSIHE z klasycznym HE.

In [ ]:
import cv2
import os
from matplotlib import pyplot as plt
import numpy as np

if not os.path.exists("jet.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/jet.bmp --no-check-certificate



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def calculate_bi_histogram_equalization(img, method='none'):
    hist, bins = np.histogram(img.flatten(), 256, [0, 256])
    if method == 'BBHE':
        # Próg podziału to średnia jasność
        lm = int(np.mean(img))
    elif method == 'DSIHE':
        cdf = hist.cumsum()
        diff = np.abs(cdf - (img.size / 2.0))
        lm = np.argmin(diff)
    elif method == 'none':
        print('Wybierz jedna z metod BBHE lub DSIHE')
        return None
    else:
        raise ValueError('Podano zla metode')

    H1 = hist[:lm+1]
    H2 = hist[lm+1:]
    
    C1 = H1.cumsum()
    C2 = H2.cumsum()
    
    C1_norm = C1 / C1.max() if C1.max() > 0 else C1
    C2_norm = C2 / C2.max() if C2.max() > 0 else C2

    C1n = lm * C1_norm
    C2n = (lm + 1) + (255 - lm - 1) * C2_norm
    
    lut = np.zeros(256, dtype=np.uint8)
    lut[:lm+1] = np.round(C1n).astype(np.uint8)
    lut[lm+1:] = np.round(C2n).astype(np.uint8)
    
    result_img = cv2.LUT(img, lut)
    
    return result_img, lm






img = cv2.imread('jet.bmp', cv2.IMREAD_GRAYSCALE)


he_img = cv2.equalizeHist(img)
    
bbhe_img, lm_bbhe = calculate_bi_histogram_equalization(img, 'BBHE')
dsihe_img, lm_dsihe = calculate_bi_histogram_equalization(img, 'DSIHE')
    
print(f"Próg podziału dla BBHE (średnia): {lm_bbhe}")
print(f"Próg podziału dla DSIHE (mediana): {lm_dsihe}")

plt.figure(figsize=(15, 10))

plt.subplot(2, 4, 1)
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.title('Oryginał')
plt.axis('off')

plt.subplot(2, 4, 5)
plt.hist(img.flatten(), 256, [0, 256], color='black')
plt.title('Histogram - Oryginał')

plt.subplot(2, 4, 2)
plt.imshow(he_img, cmap='gray', vmin=0, vmax=255)
plt.title('Klasyczne HE')
plt.axis('off')

plt.subplot(2, 4, 6)
plt.hist(he_img.flatten(), 256, [0, 256], color='black')
plt.title('Histogram - HE')

plt.subplot(2, 4, 3)
plt.imshow(bbhe_img, cmap='gray', vmin=0, vmax=255)
plt.title(f'BBHE (lm={lm_bbhe})')
plt.axis('off')

plt.subplot(2, 4, 7)
plt.hist(bbhe_img.flatten(), 256, [0, 256], color='black')
plt.title('Histogram - BBHE')

plt.subplot(2, 4, 4)
plt.imshow(dsihe_img, cmap='gray', vmin=0, vmax=255)
plt.title(f'DSIHE (lm={lm_dsihe})')
plt.axis('off')

plt.subplot(2, 4, 8)
plt.hist(dsihe_img.flatten(), 256, [0, 256], color='black')
plt.title('Histogram - DSIHE')

plt.tight_layout()
plt.show()